# LSTM + Conformal Prediction

Trains an LSTM point-prediction model per `(ticker, crisis regime)` split, wraps it with Split Conformal Prediction and Adaptive Conformal Inference (ACI), walks forward through the test window, and writes daily interval arrays to `results/` for the aggregator notebook to consume.

Two sections:
1. **Smoke test** — one split, both wrappers. Verifies wiring works and produces a sanity-checkable marginal coverage in `[0.85, 0.95]`.
2. **Full grid** — every `(ticker, regime)` split × `{Split CP, ACI}`. Gated by a `RUN_FULL_GRID` flag because it takes a while.


In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
for _p in (_here, *_here.parents):
    if (_p / "requirements.txt").exists():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break


In [ ]:
import numpy as np
import pandas as pd
import torch

from src.analysis._shared import (
    ensure_data,
    iterate_experiments,
    walk_forward,
    make_wrapper,
    save_intervals,
)
from src.data.data import Ticker
from src.models.lstm import LSTMModel


In [ ]:
NUM_LAGS = 5
CONFIDENCE_LEVEL = 0.90  # alpha = 0.10, matches Metrics.md
ACI_STEP_SIZE = 0.01

MODEL_NAME = "lstm"

np.random.seed(0)
torch.manual_seed(0)


## Load data

Downloads the full `ALL_TICKERS` panel from 2003-01-01 onward (cached under `datasets/all_tickers.csv`). Re-running is a no-op once the CSV exists.


In [ ]:
data = ensure_data()
print(f"{len(data):,} rows  |  {data.index.min().date()} -> {data.index.max().date()}")
data.head()


## Smoke test: one split

Picks the 2020 x MSFT split, runs Split CP and ACI, saves intervals, and verifies marginal coverage is in `[0.85, 0.95]`. Outside that range usually means a mismatched `confidence_level` (easy bug: `BaseConformal` defaults to 0.95, but we want 0.90).


In [ ]:
smoke_split = next(
    s for s in iterate_experiments()
    if s.regime == "2020" and s.ticker == Ticker.Microsoft
)
smoke_split


In [ ]:
train_df = smoke_split.slice_train(data)
calib_df = smoke_split.slice_calib(data)
full_df  = smoke_split.slice_full(data)

print(f"train: {len(train_df):>5} rows  |  calib: {len(calib_df):>4} rows  |  test window has {((full_df.index >= smoke_split.test_start) & (full_df.index <= smoke_split.test_end)).sum()} rows")


In [ ]:
split_wrapper = make_wrapper(
    model_class=LSTMModel,
    label_col=smoke_split.label_col,
    num_lags=NUM_LAGS,
    confidence_level=CONFIDENCE_LEVEL,
    step_size=None,
)
split_wrapper.fit_and_calibrate(train_df, calib_df)

split_intervals = walk_forward(
    split_wrapper, full_df,
    test_start=smoke_split.test_start,
    test_end=smoke_split.test_end,
    is_adaptive=False,
)
save_intervals(split_intervals, MODEL_NAME, "split", smoke_split.ticker, smoke_split.regime)

covered = (split_intervals["r"] >= split_intervals["l"]) & (split_intervals["r"] <= split_intervals["u"])
print(f"Split CP  coverage={covered.mean():.3f}  mean_width={(split_intervals['u'] - split_intervals['l']).mean():.4f}")


In [ ]:
aci_wrapper = make_wrapper(
    model_class=LSTMModel,
    label_col=smoke_split.label_col,
    num_lags=NUM_LAGS,
    confidence_level=CONFIDENCE_LEVEL,
    step_size=ACI_STEP_SIZE,
)
aci_wrapper.fit_and_calibrate(train_df, calib_df)

aci_intervals = walk_forward(
    aci_wrapper, full_df,
    test_start=smoke_split.test_start,
    test_end=smoke_split.test_end,
    is_adaptive=True,
)
save_intervals(aci_intervals, MODEL_NAME, "aci", smoke_split.ticker, smoke_split.regime)

covered = (aci_intervals["r"] >= aci_intervals["l"]) & (aci_intervals["r"] <= aci_intervals["u"])
print(f"ACI       coverage={covered.mean():.3f}  mean_width={(aci_intervals['u'] - aci_intervals['l']).mean():.4f}")


## Full grid

Loops over every `(ticker, regime)` split and runs both wrappers. Takes a while — LSTMs are trained fresh per split (~21 splits x 2 wrappers = 42 fits). Flip the flag when you're ready.


In [ ]:
RUN_FULL_GRID = False  # set True to run the full experiment

if RUN_FULL_GRID:
    MIN_TRAIN_ROWS = NUM_LAGS + 50  # skip splits with too little history (e.g., Zoom in 2008)

    for split in iterate_experiments():
        train_df = split.slice_train(data)
        calib_df = split.slice_calib(data)
        full_df  = split.slice_full(data)

        if len(train_df) < MIN_TRAIN_ROWS:
            print(f"skip {split.regime} {split.ticker.value:5s}: only {len(train_df)} train rows")
            continue

        for wrapper_name, step_size in [("split", None), ("aci", ACI_STEP_SIZE)]:
            wrapper = make_wrapper(
                model_class=LSTMModel,
                label_col=split.label_col,
                num_lags=NUM_LAGS,
                confidence_level=CONFIDENCE_LEVEL,
                step_size=step_size,
            )
            wrapper.fit_and_calibrate(train_df, calib_df)
            intervals = walk_forward(
                wrapper, full_df,
                test_start=split.test_start,
                test_end=split.test_end,
                is_adaptive=(step_size is not None),
            )
            save_intervals(intervals, MODEL_NAME, wrapper_name, split.ticker, split.regime)

            covered = (intervals["r"] >= intervals["l"]) & (intervals["r"] <= intervals["u"])
            print(f"  {split.regime} {split.ticker.value:5s} {wrapper_name:5s}: cov={covered.mean():.3f}")
